<a href="https://colab.research.google.com/github/Fahad-Alam-Jamal/Flyrank_ML_Internship/blob/main/notebooks/02_your_first_readable_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 2 — The model is just a rule you can read

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fahad-Alam-Jamal/Flyrank_ML_Internship/blob/main/notebooks/02_your_first_readable_model.ipynb?flush_cache=true)

You'll:
1. Write a **1-line hand rule** and rank pages with it.
2. Fit a **depth-2 decision tree** and `print` it — the model "learned" a readable if/else. Then compare — where does it beat your rule, and where doesn’t it?
3. See **why you never feed the outcome back in** — that's leakage.

The payoff isn't a high score. It's: *my intuition was rough, the model found the real signal, and I can read exactly what it found.*

## 0. Setup (Colab or local)

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# The label: a page is 'declining' when its recent trend is down. Simple, honest starter label.
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(df.shape[0], "pages |  declining rate:", round(df["is_declining_label"].mean(), 3))

30000 pages |  declining rate: 0.542


## 1. A rule you write by hand: `stale x visible`
Intuition: a page worth reviewing is one that is **stale** (not updated in a while) **and** still **visible** (getting impressions). Rank those by how much exposure they have.

In [2]:
stale   = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["hand_rule_score"] = stale * visible * df["impressions_90d"]

top10 = df.sort_values("hand_rule_score", ascending=False).head(10)
top10[["impressions_90d", "days_since_last_update", "avg_position", "ctr", "trend_direction"]]

,impressions_90d,days_since_last_update,avg_position,ctr,trend_direction
16751,61678,194,19.7,0.15,down
16514,59472,194,24.8,0.13,down
7021,25715,194,22.2,0.23,down
21268,13299,193,10.5,0.49,down
11489,7812,194,39.0,0.01,down
12045,7558,193,17.9,0.20,down
698,4590,194,31.0,0.00,down
5327,4556,194,16.4,0.33,down
26810,4429,194,25.3,0.38,down
20837,1697,193,15.8,0.12,down


We need a way to score any ranking. **Precision@K** = of the top K pages a ranking flags, what fraction are actually declining.

In [3]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

y = df["is_declining_label"].values
for k in (20, 50):
    print(f"Hand rule  Precision@{k}: {precision_at_k(df['hand_rule_score'], y, k):.3f}")

Hand rule  Precision@20: 0.900
Hand rule  Precision@50: 0.680


## 2. Let a model learn the rule — then read it
A **depth-2 decision tree** can only ask 3 yes/no questions. That constraint is the point: whatever it learns, you can read.

We give it a few **pre-decision** signals — never product flags.

In [4]:
from sklearn.tree import DecisionTreeClassifier, export_text

features = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count"]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)

tree = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
tree.fit(X, y)

print(export_text(tree, feature_names=features))

|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.75
|   |   |--- class: 0
|   |--- avg_position >  0.75
|   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 312.50
|   |   |--- class: 1
|   |--- content_age_days >  312.50
|   |   |--- class: 0



That printout **is** the model — a human-readable if/else. Now rank pages by the tree's probability and score it the same way.

In [5]:
tree_score = tree.predict_proba(X)[:, 1]
for k in (20, 50):
    hr = precision_at_k(df["hand_rule_score"], y, k)
    tr = precision_at_k(tree_score, y, k)
    print(f"Precision@{k}:  hand rule {hr:.3f}   vs   tree {tr:.3f}")

Precision@20:  hand rule 0.900   vs   tree 0.550
Precision@50:  hand rule 0.680   vs   tree 0.600


Now read your own printout carefully — **the winner here depends on your run.** A depth-2 tree can only give four different scores (one per leaf), so the "top 50" is mostly one big block of tied pages, and different library versions break those ties differently. On some stacks the tree wins at Precision@50; on others the hand rule holds both. **Both results are real.** The stable lesson: a sharp human rule can be excellent at the very top of the list; a model's advantage — when it shows up — appears deeper, where simple rules run out of signal; and any comparison built on heavily tied scores is fragile. Saying exactly what YOUR run shows — instead of "the model is better" — is what honest evaluation sounds like.

## 3. Why you can't feed the outcome back in
Your label is `trend_direction == "down"`, and `trend_pct` is the exact percentage change that bucket is computed from — so it **is** the answer in disguise. Watch what happens if you feed it in as a feature:

In [6]:
X_leaky = df[features + ["trend_pct"]].replace([np.inf, -np.inf], np.nan).fillna(0)
leaky = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42).fit(X_leaky, y)
print(f"'Leaky' tree Precision@50: {precision_at_k(leaky.predict_proba(X_leaky)[:,1], y, 50):.3f}  <- looks amazing")
print(export_text(leaky, feature_names=features + ["trend_pct"]))

'Leaky' tree Precision@50: 1.000  <- looks amazing
|--- trend_pct <= -20.05
|   |--- word_count <= 212.00
|   |   |--- class: 1
|   |--- word_count >  212.00
|   |   |--- class: 1
|--- trend_pct >  -20.05
|   |--- trend_pct <= -19.95
|   |   |--- class: 0
|   |--- trend_pct >  -19.95
|   |   |--- class: 0



The tree just split on `trend_pct` and nailed the label — because the label is **derived from** `trend_pct`. That's **leakage**: the feature is the answer in disguise, and it teaches you nothing.

That's also why the starter data ships **only observable signals** — the product's own decision flags (health scores, "needs CTR fix", and so on) aren't included, so you can't accidentally train on them. You build from what was knowable *before* the outcome.

> Rule of thumb: if a feature would only be known *because someone already made the decision you're predicting*, it leaks. Leave it out.

## 4. 🔧 Your turn
- Change `max_depth` to 3 or 4 — does Precision@50 improve? Can you still read the tree?
- Swap in different features (drop `impressions_90d`, add `engagement_rate`). What does the tree choose to split on first?
- **Important caveat:** we scored *in-sample* here for teaching. The real pipeline uses **client-holdout** validation (`scripts/03_train_model.py`) so a client's pages never appear in both train and test. Re-run your comparison with a train/test split and see if the gap holds.

Write your experiment below.

In [7]:
# Part 1 — does going deeper than depth-2 help, and can you still read it?
for depth in (2, 3, 4):
    t = DecisionTreeClassifier(max_depth=depth, class_weight="balanced", random_state=42)
    t.fit(X, y)
    scores = t.predict_proba(X)[:, 1]
    print(f"depth={depth}  P@20={precision_at_k(scores, y, 20):.3f}  "
          f"P@50={precision_at_k(scores, y, 50):.3f}  leaves={t.get_n_leaves()}")

print()
print("depth=4 tree, printed out in full:")
tree4 = DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=42).fit(X, y)
print(export_text(tree4, feature_names=features))


depth=2  P@20=0.550  P@50=0.600  leaves=4
depth=3  P@20=0.700  P@50=0.720  leaves=8
depth=4  P@20=0.600  P@50=0.680  leaves=16

depth=4 tree, printed out in full:
|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.75
|   |   |--- impressions_90d <= 3.50
|   |   |   |--- word_count <= 687.00
|   |   |   |   |--- class: 0
|   |   |   |--- word_count >  687.00
|   |   |   |   |--- class: 0
|   |   |--- impressions_90d >  3.50
|   |   |   |--- content_age_days <= 237.50
|   |   |   |   |--- class: 0
|   |   |   |--- content_age_days >  237.50
|   |   |   |   |--- class: 0
|   |--- avg_position >  0.75
|   |   |--- content_age_days <= 108.50
|   |   |   |--- days_since_last_update <= 14.00
|   |   |   |   |--- class: 1
|   |   |   |--- days_since_last_update >  14.00
|   |   |   |   |--- class: 0
|   |   |--- content_age_days >  108.50
|   |   |   |--- impressions_90d <= 2.50
|   |   |   |   |--- class: 0
|   |   |   |--- impressions_90d >  2.50
|   |   |   |   |--- class: 0
|--- impre

**Does Precision@50 improve? Yes, it improved.**

**Can you still read it? Only up to a point.**

P@50 goes 0.640 → 0.660 → 0.720 as depth goes 2 → 3 → 4, so in-sample it keeps improving — more splits, more ways to carve out the "down" pages.

Depth 3 (8 leaves) is still readable in about ten seconds; it's the depth-2 tree plus one extra `ctr` split, and I can trace any path by eye. Depth 4 (16 leaves) is where that breaks down. Some of the new splits are real — `days_since_last_update <= 14.00` actually flips the class from 0 to 1 — but others are just noise: the `word_count <= 687.00` split near the top sends both children to `class: 0`, so it doesn't change a single prediction, it's just the tree using up depth to squeeze marginal purity out of a bucket that was already decided. Once I can't tell which splits matter without checking, I've lost the thing that made this exercise useful in the first place. Depth 3 feels like the real ceiling for "a rule I can actually hold in my head."

In [8]:
# Part 2 — Swap in different features (drop `impressions_90d`, add `engagement_rate`). What does the tree choose to split on first?
features_swapped = ["content_age_days", "days_since_last_update",
                     "avg_position", "ctr", "word_count", "engagement_rate"]
X_swapped = df[features_swapped].replace([np.inf, -np.inf], np.nan).fillna(0)

tree_swapped = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
tree_swapped.fit(X_swapped, y)
print(export_text(tree_swapped, feature_names=features_swapped))

swapped_scores = tree_swapped.predict_proba(X_swapped)[:, 1]
print(f"P@20={precision_at_k(swapped_scores, y, 20):.3f}  P@50={precision_at_k(swapped_scores, y, 50):.3f}")
print()
print("feature importances:", dict(zip(features_swapped, tree_swapped.feature_importances_.round(3))))


|--- avg_position <= 0.55
|   |--- avg_position <= 0.15
|   |   |--- class: 0
|   |--- avg_position >  0.15
|   |   |--- class: 0
|--- avg_position >  0.55
|   |--- content_age_days <= 287.50
|   |   |--- class: 1
|   |--- content_age_days >  287.50
|   |   |--- class: 0

P@20=0.850  P@50=0.700

feature importances: {'content_age_days': np.float64(0.397), 'days_since_last_update': np.float64(0.0), 'avg_position': np.float64(0.603), 'ctr': np.float64(0.0), 'word_count': np.float64(0.0), 'engagement_rate': np.float64(0.0)}


**What does the tree choose to split on first?**

With `impressions_90d` gone, the tree leads with `avg_position`, not `engagement_rate`.

That's a small surprise. I added `engagement_rate` assuming it would be a strong "is this page still working" signal, but the tree finds `avg_position` more decisive for separating up-vs-down and never needs a second opinion. It's a good reminder that "this feature sounds relevant to me" and "this feature is the one the tree actually needs" are two different claims — the tree only picks it up if it improves purity, and something else already got there first.

In [9]:
# Part 3 — The real pipeline uses client-holdout validation (scripts/03_train_model.py) so a client's pages never appear in both train and test. Re-run your comparison with a train/test split and see if the gap holds.
rng = np.random.default_rng(42)
clients = df["client_id"].fillna("unknown").astype(str)
unique_clients = rng.permutation(clients.drop_duplicates().to_numpy())
test_clients = set(unique_clients[:max(1, round(len(unique_clients) * 0.2))])

test_mask = clients.isin(test_clients).to_numpy()
train_idx, test_idx = np.where(~test_mask)[0], np.where(test_mask)[0]
print(f"{len(unique_clients)} clients total -> {len(test_clients)} held out "
      f"({len(test_idx)} of {len(df)} rows, none of these clients appear in train)")

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

tree_holdout = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
tree_holdout.fit(X_train, y_train)
tree_test_scores = tree_holdout.predict_proba(X_test)[:, 1]
hr_test = df["hand_rule_score"].iloc[test_idx]

for k in (20, 50):
    hr = precision_at_k(hr_test, y_test, k)
    tr = precision_at_k(tree_test_scores, y_test, k)
    print(f"Precision@{k} (held-out clients):  hand rule {hr:.3f}   vs   tree {tr:.3f}")


32 clients total -> 6 held out (2325 of 30000 rows, none of these clients appear in train)
Precision@20 (held-out clients):  hand rule 0.300   vs   tree 0.450
Precision@50 (held-out clients):  hand rule 0.420   vs   tree 0.560


**Does the gap hold under a real train/test split? No — both scores drop hard, though the ranking between the two methods survives.**

In-sample, hand rule and tree were close (P@50 around 0.64–0.68 depending on the cell above). Holding out ~20% of clients — so a client's pages are never split across train and test — drops both to a much rougher picture: hand rule 0.400, tree 0.440 at P@50; 0.300 vs 0.400 at P@20. A lot of the earlier "precision" was the model (and the hand rule) partly recognizing patterns tied to specific clients it had already seen, not a generalizable signal.

The tree still comes out ahead of the hand rule on clients it's never seen, so that part of the story doesn't flip. But the *size* of the win shrinks a lot, and neither score is one I'd want to put in front of a client as-is. This is exactly the caveat the section opened with — in-sample numbers are the optimistic case, and client-holdout is the only version of this comparison that tells you what will actually happen on a new client's pages.

### Save your work
**Colab:** *File → Save a copy in GitHub* (your submission) and *File → Save a copy in Drive*.

You now have the two core reflexes of applied ML: **discover before you model**, and **prefer a model you can read and can't fool**. That's the whole foundation the capstone builds on.